Speech Controlled Maze Navigation Demo

Goal: Use spoken commands to guide the blue player to the green goal.

Commands: `up`, `down`, `left`, `right`, `stop`, `go`

- `stop` pauses movement
- `go` resumes movement

## 1. Imports

In [1]:
import os
import json
import pickle
import time
from dataclasses import dataclass
from typing import Optional

import numpy as np
import librosa
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.utils import register_keras_serializable

try:
    import sounddevice as sd
except Exception as e:
    sd = None
    print('Microphone unavailable:', e)

from IPython.display import clear_output, display, HTML

print('TensorFlow:', tf.__version__)

TensorFlow: 2.21.0


## 2. Custom Keras objects

In [2]:
@register_keras_serializable(package='CustomSchedules')
class CosineDecayWithWarmup(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_learning_rate=1e-3, decay_steps=10000,
                 warmup_steps=1000, min_learning_rate=1e-6):
        super().__init__()
        self.initial_learning_rate = initial_learning_rate
        self.decay_steps = decay_steps
        self.warmup_steps = warmup_steps
        self.min_learning_rate = min_learning_rate

    def __call__(self, step):
        step = tf.cast(step, tf.float32)
        warmup_f = tf.cast(self.warmup_steps, tf.float32)
        decay_f = tf.cast(self.decay_steps, tf.float32)
        warmup_lr = self.initial_learning_rate * (step / tf.maximum(warmup_f, 1.0))
        decay_step = step - warmup_f
        total_decay = tf.maximum(decay_f - warmup_f, 1.0)
        cosine = 0.5 * (1.0 + tf.cos(np.pi * decay_step / total_decay))
        decayed_lr = self.min_learning_rate + (self.initial_learning_rate - self.min_learning_rate) * cosine
        return tf.where(step < warmup_f, warmup_lr, decayed_lr)

    def get_config(self):
        return {
            'initial_learning_rate': float(self.initial_learning_rate),
            'decay_steps': int(self.decay_steps),
            'warmup_steps': int(self.warmup_steps),
            'min_learning_rate': float(self.min_learning_rate),
        }


def masked_sparse_crossentropy(y_true, y_pred):
    mask = tf.cast(tf.not_equal(y_true, -1), tf.float32)
    y_true = tf.maximum(y_true, 0)
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_sum(loss * mask) / (tf.reduce_sum(mask) + 1e-8)

CUSTOM_OBJECTS = {
    'CosineDecayWithWarmup': CosineDecayWithWarmup,
    'masked_sparse_crossentropy': masked_sparse_crossentropy,
}

print('Custom objects ready.')

Custom objects ready.


## 3. Model artifact paths

In [3]:
MODEL_DIR = './data'
MODEL_FILE = 'resnet_sequencing_final.keras'
LABEL_ENCODER_FILE = 'label_encoder.pkl'
CONFIG_FILE = 'model_config.json'

model_path = os.path.join(MODEL_DIR, MODEL_FILE)
label_encoder_path = os.path.join(MODEL_DIR, LABEL_ENCODER_FILE)
config_path = os.path.join(MODEL_DIR, CONFIG_FILE)

print('Current directory:', os.getcwd())
print('Files in MODEL_DIR:', os.listdir(MODEL_DIR) if os.path.exists(MODEL_DIR) else 'MODEL_DIR not found')
print('\nChecks:')
print('Model:', os.path.exists(model_path), model_path)
print('Label encoder:', os.path.exists(label_encoder_path), label_encoder_path)
print('Config:', os.path.exists(config_path), config_path)

Current directory: /Users/vishalvinjamuri/Downloads/ML
Files in MODEL_DIR: ['label_encoder.pkl', 'resnet_sequencing_final.keras', 'model_config.json']

Checks:
Model: True ./data/resnet_sequencing_final.keras
Label encoder: True ./data/label_encoder.pkl
Config: True ./data/model_config.json


## 4. Model adapter

In [4]:
class KerasSpeechCommandAdapter:
    def __init__(self, model_path, label_encoder_path, config_path):
        with open(config_path, 'r') as f:
            self.config = json.load(f)
        with open(label_encoder_path, 'rb') as f:
            self.label_encoder = pickle.load(f)

        self.model = keras.models.load_model(
            model_path,
            custom_objects=CUSTOM_OBJECTS,
            compile=False,
        )

        self.sample_rate = int(self.config.get('sample_rate', 16000))
        self.audio_length = int(self.config.get('audio_length', 16000))
        self.n_mels = int(self.config.get('n_mels', 128))
        self.n_fft = int(self.config.get('n_fft', 2048))
        self.hop_length = int(self.config.get('hop_length', 160))
        self.mean = float(self.config.get('single_mean', 0.0))
        self.std = float(self.config.get('single_std', 1.0))

    def preprocess_array(self, audio):
        if len(audio) < self.audio_length:
            audio = np.pad(audio, (0, self.audio_length - len(audio)), mode='constant')
        else:
            audio = audio[:self.audio_length]

        max_val = np.max(np.abs(audio))
        if max_val > 0:
            audio = audio / max_val

        mel = librosa.feature.melspectrogram(
            y=audio,
            sr=self.sample_rate,
            n_mels=self.n_mels,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        log_mel = (log_mel - self.mean) / (self.std + 1e-8)
        return log_mel[np.newaxis, ..., np.newaxis].astype(np.float32)

    def predict_from_array(self, audio):
        x = self.preprocess_array(audio)
        outputs = self.model.predict(x, verbose=0)
        return self._decode_outputs(outputs)

    def _decode_outputs(self, outputs):
        if isinstance(outputs, list):
            probs = outputs[0][0]
        elif isinstance(outputs, dict):
            probs = outputs.get('single_command', list(outputs.values())[0])[0]
        else:
            probs = outputs[0]

        allowed = {"up", "down", "left", "right", "stop", "go"}
        classes = list(self.label_encoder.classes_)

        masked_probs = probs.copy()

        for i, cls in enumerate(classes):
            if cls not in allowed:
                masked_probs[i] = 0.0

        pred_idx = int(np.argmax(masked_probs))
        confidence = float(masked_probs[pred_idx])
        label = self.label_encoder.inverse_transform([pred_idx])[0]

        return label, confidence, probs


if os.path.exists(model_path) and os.path.exists(label_encoder_path) and os.path.exists(config_path):
    adapter = KerasSpeechCommandAdapter(model_path, label_encoder_path, config_path)
    print('Model loaded successfully.')
    print('Classes:', list(adapter.label_encoder.classes_))
else:
    adapter = None
    print('Missing artifacts. Fix Section 3 paths first.')

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model loaded successfully.
Classes: [np.str_('_background_noise_'), np.str_('down'), np.str_('go'), np.str_('left'), np.str_('no'), np.str_('off'), np.str_('on'), np.str_('right'), np.str_('stop'), np.str_('up'), np.str_('yes')]


## 5. Audio recording helper

In [5]:
def record_seconds(seconds=1.0, sample_rate=16000):
    if sd is None:
        raise RuntimeError('sounddevice is unavailable')
    audio = sd.rec(int(seconds * sample_rate), samplerate=sample_rate, channels=1, dtype='float32')
    sd.wait()
    return audio.squeeze()

## 6. Clean aesthetic maze UI

In [6]:
MAZE_TEMPLATE = [
    list('111111111111'),
    list('1S0000000001'),
    list('101111011101'),
    list('100001010001'),
    list('111101010111'),
    list('100001000001'),
    list('101111111101'),
    list('1000000000G1'),
    list('111111111111'),
]

GAME_COMMANDS = {'up', 'down', 'left', 'right', 'stop', 'go'}
MOVE_COMMANDS = {'up', 'down', 'left', 'right'}

@dataclass
class DemoStats:
    total_predictions: int = 0
    accepted_commands: int = 0
    ignored_low_confidence: int = 0
    ignored_non_game: int = 0
    movement_commands: int = 0
    valid_moves: int = 0
    wall_collisions: int = 0
    stop_commands: int = 0
    go_commands: int = 0
    start_time: float = 0.0
    end_time: Optional[float] = None
    reached_goal: bool = False

    def elapsed(self):
        return (self.end_time or time.time()) - self.start_time


class AestheticVoiceMaze:
    def __init__(self, adapter=None, confidence_threshold=0.50):
        self.adapter = adapter
        self.confidence_threshold = confidence_threshold
        self.maze = [row.copy() for row in MAZE_TEMPLATE]
        self.pos = self.find_cell('S')
        self.goal = self.find_cell('G')
        self.paused = False
        self.last_command = 'None'
        self.last_confidence = 0.0
        self.history = []
        self.current_window = 0
        self.total_windows = 0
        self.stats = DemoStats(start_time=time.time())

    def find_cell(self, target):
        for r, row in enumerate(self.maze):
            for c, val in enumerate(row):
                if val == target:
                    return (r, c)
        raise ValueError(f'{target} not found')

    def maze_html(self):
        cells = []
        for r, row in enumerate(self.maze):
            for c, val in enumerate(row):
                cls = 'path'
                content = ''
                if val == '1':
                    cls = 'wall'
                elif val == 'G':
                    cls = 'goal'
                    content = '★'
                elif val == 'S':
                    cls = 'start'
                if (r, c) == self.pos:
                    cls = 'player'
                    content = '●'
                cells.append(f'<div class="cell {cls}">{content}</div>')
        return ''.join(cells)

    def render(self, message=''):
        clear_output(wait=True)
        progress = 0 if self.total_windows == 0 else int(100 * self.current_window / self.total_windows)
        accepted_rate = 0 if self.stats.total_predictions == 0 else self.stats.accepted_commands / self.stats.total_predictions
        move_acc = 0 if self.stats.movement_commands == 0 else self.stats.valid_moves / self.stats.movement_commands
        status_text = 'PAUSED' if self.paused else 'ACTIVE'
        status_class = 'paused' if self.paused else 'active'
        recent = ''.join([f'<li>{h}</li>' for h in self.history[-5:]]) or '<li>No predictions yet.</li>'

        html = f'''
        <style>
        .game-wrap {{
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
            max-width: 880px;
            margin: 8px 0;
            color: #111827;
        }}
        .hero {{
            background: linear-gradient(135deg, #0f172a, #1e3a8a 55%, #0284c7);
            color: white;
            border-radius: 18px;
            padding: 18px 22px;
            box-shadow: 0 14px 35px rgba(15, 23, 42, 0.25);
            margin-bottom: 16px;
        }}
        .hero h1 {{ margin: 0 0 6px; font-size: 28px; }}
        .hero p {{ margin: 0; opacity: 0.92; font-size: 14px; }}
        .layout {{ display: flex; gap: 18px; align-items: flex-start; }}
        .maze-card, .panel {{
            background: #f8fafc;
            border: 1px solid #e5e7eb;
            border-radius: 18px;
            padding: 16px;
            box-shadow: 0 10px 25px rgba(15, 23, 42, 0.08);
        }}
        .maze {{
            display: grid;
            grid-template-columns: repeat(12, 34px);
            grid-template-rows: repeat(9, 34px);
            gap: 4px;
            background: #e2e8f0;
            padding: 10px;
            border-radius: 16px;
        }}
        .cell {{
            width: 34px; height: 34px;
            border-radius: 9px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-weight: 800;
            font-size: 20px;
        }}
        .wall {{ background: #111827; box-shadow: inset 0 0 0 1px #374151; }}
        .path {{ background: #ffffff; }}
        .start {{ background: #dbeafe; }}
        .goal {{ background: #22c55e; color: white; }}
        .player {{ background: #2563eb; color: white; box-shadow: 0 0 0 4px rgba(37, 99, 235, 0.22); }}
        .panel {{ flex: 1; min-width: 310px; }}
        .pill {{
            display: inline-block;
            padding: 5px 10px;
            border-radius: 999px;
            font-size: 12px;
            font-weight: 700;
            margin-right: 6px;
        }}
        .active {{ background: #dcfce7; color: #166534; }}
        .paused {{ background: #fee2e2; color: #991b1b; }}
        .metric-grid {{ display: grid; grid-template-columns: repeat(2, 1fr); gap: 8px; margin-top: 12px; }}
        .metric {{ background: white; border: 1px solid #e5e7eb; border-radius: 12px; padding: 10px; }}
        .metric b {{ display: block; font-size: 18px; margin-top: 2px; }}
        .bar {{ height: 10px; background: #e5e7eb; border-radius: 999px; overflow: hidden; margin: 12px 0; }}
        .bar-inner {{ height: 100%; width: {progress}%; background: linear-gradient(90deg, #38bdf8, #2563eb); }}
        .commands {{ display: flex; flex-wrap: wrap; gap: 6px; margin-top: 10px; }}
        .cmd {{ background: #e0f2fe; color: #075985; border-radius: 10px; padding: 5px 9px; font-weight: 700; font-size: 12px; }}
        .history {{ margin-top: 12px; font-size: 13px; }}
        .history ul {{ margin: 6px 0 0 18px; padding: 0; }}
        .message {{ margin-top: 10px; color: #475569; font-size: 13px; }}
        </style>

        <div class="game-wrap">
          <div class="hero">
            <h1>Voice controlled Maze</h1>
            <p>Say commands to move the blue player to the goal Use <b>stop</b> to pause and <b>go</b> to continue</p>
          </div>
          <div class="layout">
            <div class="maze-card">
              <div class="maze">{self.maze_html()}</div>
            </div>
            <div class="panel">
              <span class="pill {status_class}">{status_text}</span>
              <span class="pill" style="background:#ede9fe;color:#5b21b6;">threshold {self.confidence_threshold:.2f}</span>
              <h2 style="margin:12px 0 4px;">Last prediction</h2>
              <div style="font-size:26px;font-weight:800;">{self.last_command}</div>
              <div style="color:#64748b;">confidence: {self.last_confidence:.3f}</div>
              <div class="bar"><div class="bar-inner"></div></div>
              <div style="font-size:12px;color:#64748b;">Live run progress: {self.current_window}/{self.total_windows}</div>
              <div class="commands">
                <span class="cmd">up</span><span class="cmd">down</span><span class="cmd">left</span><span class="cmd">right</span><span class="cmd">stop</span><span class="cmd">go</span>
              </div>
              <div class="metric-grid">
                <div class="metric">Valid moves<b>{self.stats.valid_moves}</b></div>
                <div class="metric">Collisions<b>{self.stats.wall_collisions}</b></div>
                <div class="metric">Accepted rate<b>{accepted_rate:.2f}</b></div>
                <div class="metric">Move accuracy<b>{move_acc:.2f}</b></div>
              </div>
              <div class="history"><b>Recent events</b><ul>{recent}</ul></div>
              <div class="message">{message}</div>
            </div>
          </div>
        </div>
        '''
        display(HTML(html))

    def apply_command(self, command, confidence=1.0):
        self.last_command = str(command)
        self.last_confidence = float(confidence)

        if command not in GAME_COMMANDS:
            self.stats.ignored_non_game += 1
            self.history.append(f'Ignored non-game command: {command}')
            return

        if confidence < self.confidence_threshold:
            self.stats.ignored_low_confidence += 1
            self.history.append(f'Ignored low confidence: {command} ({confidence:.2f})')
            return

        self.stats.accepted_commands += 1
        self.history.append(f'Accepted: {command} ({confidence:.2f})')

        if command == 'stop':
            self.paused = True
            self.stats.stop_commands += 1
            return

        if command == 'go':
            self.paused = False
            self.stats.go_commands += 1
            return

        if self.paused:
            self.history.append('Paused: movement ignored')
            return

        if command in MOVE_COMMANDS:
            self.stats.movement_commands += 1
            dr, dc = {'up': (-1, 0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}[command]
            r, c = self.pos
            nr, nc = r + dr, c + dc

            if self.maze[nr][nc] == '1':
                self.stats.wall_collisions += 1
                self.history.append('Wall collision')
            else:
                self.pos = (nr, nc)
                self.stats.valid_moves += 1

        if self.pos == self.goal:
            self.stats.reached_goal = True
            self.stats.end_time = time.time()
            self.history.append('Goal reached!')

    def manual_step(self, command):
        self.apply_command(command, 1.0)
        self.render('Manual test mode.')

    def live_step(self):
        if self.adapter is None:
            raise RuntimeError('No model adapter loaded.')
        audio = record_seconds(1.0, self.adapter.sample_rate)
        command, confidence, _ = self.adapter.predict_from_array(audio)
        self.stats.total_predictions += 1
        self.apply_command(command, confidence)
        self.render('Single live prediction complete.')

    def live_run(self, window_seconds=0.25, pause_between_windows=0.08):
        if self.adapter is None:
            raise RuntimeError('No model adapter loaded.')

        if sd is None:
            raise RuntimeError('sounddevice is unavailable.')

        self.current_window = 0
        self.total_windows = 999999

        self.render('Live mode started. Reach the goal to finish.')
        time.sleep(1.0)

        while not self.stats.reached_goal:
            self.current_window += 1

            audio = record_seconds(window_seconds, self.adapter.sample_rate)


            volume = np.mean(np.abs(audio))

            if volume < 0.0001:
                self.render('Listening... (silence detected) volume = {volume:.5f}')
                continue

            command, confidence, _ = self.adapter.predict_from_array(audio)

            # Confidence filtering
            if confidence < self.confidence_threshold:
                self.render(
                    f'Ignored low confidence prediction: '
                    f'{command} ({confidence:.2f})'
                )
                continue

            self.stats.total_predictions += 1

            self.apply_command(command, confidence)

            self.render(
                f'Prediction #{self.current_window}: '
                f'{command} ({confidence:.2f})'
            )

            time.sleep(pause_between_windows)

        self.render(' Goal reached!')
        self.report()
    
    def report(self):
        move_acc = 0 if self.stats.movement_commands == 0 else self.stats.valid_moves / self.stats.movement_commands
        accepted_rate = 0 if self.stats.total_predictions == 0 else self.stats.accepted_commands / self.stats.total_predictions
        print('\n' + '='*60)
        print('VOICE MAZE FINAL REPORT')
        print('='*60)
        print(f'Reached goal:              {self.stats.reached_goal}')
        print(f'Elapsed time:              {self.stats.elapsed():.2f}s')
        print(f'Total predictions:         {self.stats.total_predictions}')
        print(f'Accepted commands:         {self.stats.accepted_commands}')
        print(f'Accepted command rate:     {accepted_rate:.3f}')
        print(f'Ignored low confidence:    {self.stats.ignored_low_confidence}')
        print(f'Ignored non-game commands: {self.stats.ignored_non_game}')
        print(f'Movement commands:         {self.stats.movement_commands}')
        print(f'Valid moves:               {self.stats.valid_moves}')
        print(f'Wall collisions:           {self.stats.wall_collisions}')
        print(f'Stop commands:             {self.stats.stop_commands}')
        print(f'Go commands:               {self.stats.go_commands}')
        print(f'In-game movement accuracy: {move_acc:.3f}')
        print('='*60)

### Full live demo
Speak one command per second. Example: `right`, pause, `right`, pause, `down`, pause...

In [7]:
game = AestheticVoiceMaze(adapter=adapter, confidence_threshold=0.50)
game.live_run(window_seconds=0.5)


VOICE MAZE FINAL REPORT
Reached goal:              True
Elapsed time:              96.83s
Total predictions:         23
Accepted commands:         23
Accepted command rate:     1.000
Ignored low confidence:    0
Ignored non-game commands: 0
Movement commands:         23
Valid moves:               21
Wall collisions:           2
Stop commands:             0
Go commands:               0
In-game movement accuracy: 0.913
